In [ ]:
!git clone https://github.com/HoangVo-Prog/MaskReconstructContrastive.git
%cd /kaggle/working/MaskReconstructContrastive/unet/src

In [ ]:
# ============================================
# Precompute ADNI slices with full preprocessing
# Output: /kaggle/working/adni_preproc/
#   - images/*.png
#   - meta.csv
# ============================================
%cd /kaggle/working/MaskReconstructContrastive/unet/src
import os
import csv
from pathlib import Path
from types import SimpleNamespace

import torch
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image
from tqdm.auto import tqdm

# Import từ code của bạn
from alzheimer_unet_data import AdniNiftiSliceDataset
%cd /kaggle/working/MaskReconstructContrastive/unet/src/v3
from train import preprocess_batch


# --------------- CONFIG ---------------

# Sửa path này cho đúng dataset ADNI trên Kaggle
# Ví dụ: "/kaggle/input/adni-nii"
ADNI_ROOT = Path("/kaggle/input/adni-10-samples/MRI samples")

# Nơi lưu dataset đã preprocess (sẽ nằm trong Output của Notebook)
OUT_ROOT = Path("/kaggle/working/adni_axial")

IMAGE_SIZE = 128          # nên set giống lúc train
APPLY_UNSHARP = True      # giống setting bạn dùng khi train

BATCH_SIZE = 16           # để 8, 16 cho nhẹ RAM
NUM_WORKERS = 0           # 0 để tận dụng cache volume tuần tự


# --------------- CHECK PATH ---------------

if not ADNI_ROOT.exists():
    raise FileNotFoundError(f"Không tìm thấy ADNI_ROOT: {ADNI_ROOT}")

OUT_ROOT.mkdir(parents=True, exist_ok=True)
IMG_DIR = OUT_ROOT / "images"
IMG_DIR.mkdir(parents=True, exist_ok=True)

print("ADNI_ROOT :", ADNI_ROOT)
print("OUT_ROOT  :", OUT_ROOT)


# --------------- ARGS CHO preprocess_batch ---------------

# Args này mô phỏng đúng các flag bạn dùng trong train.py
# Nếu lúc train bạn tắt bớt cái nào thì sửa cho khớp
pre_args = SimpleNamespace(
    pre_bias=True,         # bật bias_field_lite
    pre_norm=True,         # normalize theo brain mask
    pre_crop=True,         # tight_crop_and_resize về image_size
    pre_align=True,        # align_midline
    image_size=IMAGE_SIZE,
)


# --------------- BUILD DATASET ADNI GỐC ---------------

adni_ds = AdniNiftiSliceDataset(
    root_dir=str(ADNI_ROOT),
    image_size=IMAGE_SIZE,
    apply_unsharp=APPLY_UNSHARP,
    adni_image_type="axial",      # hoặc coronal, sagittal tùy bạn
    adni_series_filter=None,      # nếu cần lọc tên series thì truyền list substring
    adni_label_csv=None,          # nếu có file csv label thì truyền path vào đây
    middle_frac=0.4,              # giữ vùng giữa volume
    middle_subsample=1,           # dùng tất cả slice trong middle
)

print(f"ADNI total slices (dataset len): {len(adni_ds)}")

adni_loader = DataLoader(
    adni_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,          # rất quan trọng để tận dụng cache volume tuần tự
    num_workers=NUM_WORKERS,
    pin_memory=False,
    drop_last=False,
)


# --------------- LOOP VÀ LƯU RA PNG + CSV ---------------

meta_path = OUT_ROOT / "meta.csv"
num_saved = 0

with open(meta_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "filename", "label", "orig_path", "slice_idx"])

    for batch in tqdm(adni_loader, desc="Precomputing ADNI slices"):
        # batch["input"] shape: [B, 1, H, W], đã qua unsharp và volume normalize
        x = batch["input"]              # ở đây để trên CPU
        x = preprocess_batch(x, pre_args)   # áp dụng full preprocessing như train

        labels = batch["label"]         # [B]
        paths = batch["path"]           # list[str]
        sidxs = batch["slice_idx"]      # [B]

        B = x.size(0)
        for i in range(B):
            img_tensor = x[i]           # [1, H, W]
            pil = to_pil_image(img_tensor)   # grayscale PIL Image

            fname = f"{num_saved:06d}.png"
            pil.save(IMG_DIR / fname)

            writer.writerow([
                num_saved,
                fname,
                int(labels[i].item()),
                paths[i],
                int(sidxs[i].item()),
            ])

            num_saved += 1

print("Done.")
print("Total slices saved:", num_saved)
print("Images dir:", IMG_DIR)
print("Meta CSV  :", meta_path)


In [ ]:
# !rm -rf /kaggle/working/MaskReconstructContrastive